结合两个agent的最终

In [1]:
# ========================================
# StockAnalysisAgent（多智能体股票分析系统）
# ========================================

"""
# 项目简介
本项目是一个基于多智能体协作（Multi-Agent）与大语言模型的 A 股客观研究报告生成系统。
采用 Master-Worker 架构：
- 主控 Agent (StockAnalysisMaster)：负责接收用户请求，并发调度底层两个子 Agent，并统筹 Prompt 生成最终研报。
- 行情采集 Agent (MarketDataCollectorAgent)：负责获取并清洗 K 线、基本面、估值、技术面数据。
- 舆情分析 Agent (NewsSentimentAgent)：负责爬取新闻，并调用大模型进行情感极性分析。

系统严格遵守“Plan-and-Solve”范式，输出结构固定、拒绝预测后市、仅供参考的学术研报。

作者信息：
- 姓名: XXX
- GitHub: @XXX
- 日期: 2025-XX-XX
"""

# ========================================
# 第2部分：环境配置
# ========================================

# 如未安装依赖，可在 PyCharm Terminal 中执行：
# pip install -q openai python-dotenv akshare pandas pydantic requests urllib3 httpx langchain-core langchain-openai langgraph

import os
import sys
import json
import traceback
from dotenv import load_dotenv


def find_project_root(start_dir: str) -> str:
    """
    从当前脚本目录向上查找项目根目录。
    判断标准：该目录下存在 src 文件夹。
    如果你的项目结构是：
        project_root/
            src/
                agent/
                    master_agent.py
            run_stock_analysis.py
    则 project_root 会被正确找到。
    """
    current = os.path.abspath(start_dir)
    for _ in range(8):  # 最多向上找 8 层，避免死循环
        if os.path.isdir(os.path.join(current, "src")):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    return os.path.abspath(start_dir)


# ================= 1. 解决路径问题 =================
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

PROJECT_ROOT = find_project_root(current_dir)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ 当前脚本目录: {current_dir}")
print(f"✅ 项目根目录已配置: {PROJECT_ROOT}")

# ================= 2. 加载环境变量 =================
# 优先加载项目根目录下的 .env；如果不存在，再按默认方式查找
env_path = os.path.join(PROJECT_ROOT, ".env")
if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"✅ 已加载环境变量文件: {env_path}")
else:
    load_dotenv()
    print("⚠️ 项目根目录未找到 .env，已尝试使用默认 load_dotenv()")

print(f"✅ 主 Agent LLM 模型配置: {os.getenv('MODEL')}")


# ========================================
# 第3部分：智能体构建与工具定义
# ========================================

# 导入三个 Agent（主 Agent 与两个子 Agent）
from src.agent.master_agent import StockAnalysisMaster
from src.agent.NewsSentimentAgent.agents import NewsSentimentAgent
from src.agent.MarketDataCollectorAgent.agents import StockDataAgent


def main():
    # 实例化主控 Agent
    # 此时主 Agent 内部已经初始化了 OpenAI Client，并装配了两个子 Agent
    master = StockAnalysisMaster()

    # 【工程化防卡死技巧】
    # 舆情 Agent 需要调用大模型并发分析新闻，测试时可能会很慢。
    # 这里直接覆盖主 Agent 里的 news_agent 配置，限制只分析 2 条新闻，加快演示速度。
    master.news_agent = NewsSentimentAgent(default_limit=2)

    # ========================================
    # 第4部分：功能演示（主流程调度）
    # ========================================

    # 定义目标股票（如 600519 贵州茅台）
    target_stock = "600519"

    print(f"\n👤 启动主 Agent 分析任务，目标股票: {target_stock}")
    print("⏳ 正在并发调度子 Agent（抓取行情 + 大模型分析舆情）并生成研报，请稍候...\n")

    try:
        # 执行主 Agent 的 run 方法
        # 内部逻辑：ThreadPoolExecutor 并发调用 行情Agent 和 舆情Agent
        # -> 汇总 JSON -> 喂给大模型
        final_report = master.run(target_stock)

        # 打印最终生成的结构化学术研报
        print("\n" + "=" * 30 + " 最终学术研究报告 " + "=" * 30 + "\n")
        print(final_report)
        print("\n" + "=" * 78)

    except Exception as e:
        print(f"\n❌ 运行出错: {e}")
        # 打印详细 traceback 方便排查
        traceback.print_exc()


if __name__ == "__main__":
    main()


# ========================================
# 第5部分：性能评估（基于代码中的计时逻辑）
# ========================================

"""
根据控制台输出日志评估：
1. 并发调度耗时：
   由于采用 ThreadPoolExecutor(max_workers=2)，
   总耗时约等于较慢那个子 Agent（通常是舆情分析）的耗时，避免了串行等待。

2. LLM 生成耗时：
   主 Agent 聚合两份 JSON 数据后，由于 System Prompt 中使用了严格的
   Plan-and-Solve 约束和 JSON 代码块包裹，LLM 的首字延迟和完整生成时间
   通常在 15~30 秒之间。

3. 稳定性：
   如果新闻接口失败，主 Agent 的异常捕获机制不会崩溃，
   仍能在报告中输出行情数据。
"""


# ========================================
# 第6部分：项目总结与展望
# ========================================

"""
## 项目总结

### 实现的功能
- 多智能体并发调度：主 Agent 高效统筹数据采集和舆情分析。
- 数据清洗与结构化：利用 Pydantic 完美对齐了 AkShare 的脏数据与 LLM 的输入需求。
- 防幻觉机制：通过极其严苛的 System Prompt（禁止预测涨跌、强制附免责声明）
  保证了学术研究的严谨性。
- 多级容错：子 Agent 内置了反爬重试、代理隔离、数据源回退等机制。

### 遇到的挑战与解决方案
- 挑战1：网络请求被风控或代理拦截。
  - 解决方案：Agent1 引入了伪装 UA 和重试 Session；
    Agent2 使用 httpx.Client(trust_env=False) 强制绕过系统代理。

- 挑战2：大模型输出格式跑偏（如把 JSON 用 Markdown 包裹）。
  - 解决方案：编写了 _clean_json_response 清洗器，
    截取第一个 { 到最后一个 } 之间的内容，彻底解决解析报错。

- 挑战3：并发请求导致 API 限流。
  - 解决方案：在舆情并发分析时，将线程池并发度限制为 3，
    并在异常捕获中针对 RateLimitError 实现 5s、10s 的递增退避。

### 未来改进方向
- [ ] 引入 MCP (Model Context Protocol)：
      目前主 Agent 是直接通过 Python 函数调用子 Agent，
      未来可以将其改造为标准 MCP Server，实现跨语言、跨进程的灵活插拔。
- [ ] 增加 RAG 检索：
      目前财报分析基于现成指标，未来可对长财报文档建立向量索引，
      实现更深入的问答。
- [ ] 异步化重构：
      将主 Agent 的 ThreadPoolExecutor 升级为 asyncio，
      进一步提升高并发下的资源利用率。
"""

✅ 当前脚本目录: B:\桌面\exercise\agent\IUING2059-StockAnalysisAgent\hello-agents\Co-creation-projects\IUING2059-StockAnalysisAgent\src\agent\master_agent
✅ 项目根目录已配置: B:\桌面\exercise\agent\IUING2059-StockAnalysisAgent\hello-agents\Co-creation-projects\IUING2059-StockAnalysisAgent
✅ 已加载环境变量文件: B:\桌面\exercise\agent\IUING2059-StockAnalysisAgent\hello-agents\Co-creation-projects\IUING2059-StockAnalysisAgent\.env
✅ 主 Agent LLM 模型配置: None

👤 启动主 Agent 分析任务，目标股票: 600519
⏳ 正在并发调度子 Agent（抓取行情 + 大模型分析舆情）并生成研报，请稍候...


[StockAnalysisMaster] 🚀 启动综合分析任务，目标股票: 600519
[StockAnalysisMaster] 正在并发调度子 Agent...

[NewsSentimentAgent] 🚀 启动任务: 正在分析 600519 的新闻舆情 (Limit=5)...
[Cache Hit] 发现今日缓存，直接从文件读取: ./news_data_cache\600519_news_2026-10-26.json
开始并发分析 5 条新闻 (并发度: 3)...
  [LLM耗时] 2.11s -> 强势股追踪 主力资金连续5日净
  [LLM耗时] 2.60s -> 氧化锆概念带火股价，MLCC人
  [LLM耗时] 2.05s -> 10月20日13只股盘后交易额
  [LLM耗时] 7.39s -> 10月21日13只股盘后交易额
  [LLM耗时] 6.45s -> 主力动向：10月23日特大单净
分析完成！成功: 5 条，失败: 0 条。整体舆情: 中性
⏱️ 子 Agent2 内部总耗时: 9.05 秒
[NewsSentimentAgent]

'\n## 项目总结\n\n### 实现的功能\n- 多智能体并发调度：主 Agent 高效统筹数据采集和舆情分析。\n- 数据清洗与结构化：利用 Pydantic 完美对齐了 AkShare 的脏数据与 LLM 的输入需求。\n- 防幻觉机制：通过极其严苛的 System Prompt（禁止预测涨跌、强制附免责声明）\n  保证了学术研究的严谨性。\n- 多级容错：子 Agent 内置了反爬重试、代理隔离、数据源回退等机制。\n\n### 遇到的挑战与解决方案\n- 挑战1：网络请求被风控或代理拦截。\n  - 解决方案：Agent1 引入了伪装 UA 和重试 Session；\n    Agent2 使用 httpx.Client(trust_env=False) 强制绕过系统代理。\n\n- 挑战2：大模型输出格式跑偏（如把 JSON 用 Markdown 包裹）。\n  - 解决方案：编写了 _clean_json_response 清洗器，\n    截取第一个 { 到最后一个 } 之间的内容，彻底解决解析报错。\n\n- 挑战3：并发请求导致 API 限流。\n  - 解决方案：在舆情并发分析时，将线程池并发度限制为 3，\n    并在异常捕获中针对 RateLimitError 实现 5s、10s 的递增退避。\n\n### 未来改进方向\n- [ ] 引入 MCP (Model Context Protocol)：\n      目前主 Agent 是直接通过 Python 函数调用子 Agent，\n      未来可以将其改造为标准 MCP Server，实现跨语言、跨进程的灵活插拔。\n- [ ] 增加 RAG 检索：\n      目前财报分析基于现成指标，未来可对长财报文档建立向量索引，\n      实现更深入的问答。\n- [ ] 异步化重构：\n      将主 Agent 的 ThreadPoolExecutor 升级为 asyncio，\n      进一步提升高并发下的资源利用率。\n'